# Bone Fracture Detection — обучение Fast и Accurate

Ноутбук скачивает публичный Kaggle-датасет, выбирает YOLO detection-разметку, обучает две модели, считает финальные метрики и сохраняет всё в Google Drive. Перед запуском: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "kaggle>=1.6.17,<3" pyyaml pandas

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не включён: выберите T4 GPU в настройках Runtime'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/bone-fracture-detector')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Результаты:', OUTPUT_ROOT)

In [ ]:
import subprocess
DATA_ROOT = Path('/content/data/bone-fracture')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not list(DATA_ROOT.rglob('data.yaml')):
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '--dataset', 'pkdarabi/bone-fracture-detection-computer-vision-project',
        '--path', str(DATA_ROOT), '--unzip'
    ], check=True)
print('YAML-файлы:', *DATA_ROOT.rglob('data.yaml'), sep='\n')

In [ ]:
import yaml

def annotation_score(data_yaml: Path) -> int:
    score = 0
    for label in list(data_yaml.parent.rglob('labels/*.txt'))[:200]:
        lines = [line for line in label.read_text(errors='ignore').splitlines() if line.strip()]
        if lines:
            field_count = len(lines[0].split())
            if field_count == 5 or (field_count >= 7 and field_count % 2 == 1):
                score += 1
    return score

candidates = list(DATA_ROOT.rglob('data.yaml'))
assert candidates, 'data.yaml не найден'
source_yaml = max(candidates, key=annotation_score)
assert annotation_score(source_yaml) > 0, 'YOLO-разметка не найдена'
source_config = yaml.safe_load(source_yaml.read_text())
# Продуктовая постановка: локализуем любую область перелома одним классом.
# Это обосновано сильным дисбалансом исходных анатомических классов: один из них
# имеет только 3 train-примера и отсутствует в validation/test.
SINGLE_CLASS = True
SOURCE_ROOT = source_yaml.parent
DETECT_ROOT = Path('/content/data/bone-fracture-detect')

def polygon_to_box(line: str) -> str:
    fields = line.split()
    class_id = 0 if SINGLE_CLASS else int(fields[0])
    if len(fields) == 5:
        return ' '.join((str(class_id), *fields[1:]))
    coordinates = list(map(float, fields[1:]))
    assert len(fields) >= 7 and len(coordinates) % 2 == 0
    xs, ys = coordinates[0::2], coordinates[1::2]
    x_min, x_max, y_min, y_max = min(xs), max(xs), min(ys), max(ys)
    return f'{class_id} {(x_min+x_max)/2:.8f} {(y_min+y_max)/2:.8f} {x_max-x_min:.8f} {y_max-y_min:.8f}'

for directory_name in ('train', 'valid', 'test'):
    source_images = SOURCE_ROOT / directory_name / 'images'
    if not source_images.is_dir():
        continue
    target_split = DETECT_ROOT / directory_name
    target_split.mkdir(parents=True, exist_ok=True)
    target_images = target_split / 'images'
    if not target_images.exists():
        target_images.symlink_to(source_images, target_is_directory=True)
    target_labels = target_split / 'labels'
    target_labels.mkdir(exist_ok=True)
    for label in (SOURCE_ROOT / directory_name / 'labels').glob('*.txt'):
        rows = [polygon_to_box(line) for line in label.read_text().splitlines() if line.strip()]
        (target_labels / label.name).write_text('\n'.join(rows) + ('\n' if rows else ''))

output_names = ['fracture'] if SINGLE_CLASS else source_config['names']
config = {
    'path': str(DETECT_ROOT), 'train': 'train/images', 'val': 'valid/images',
    'test': 'test/images', 'nc': len(output_names), 'names': output_names
}
DATA_YAML = DETECT_ROOT / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print('Выбран исходный YAML:', source_yaml)
print(DATA_YAML.read_text())

## Минимальный аудит данных

In [ ]:
from collections import Counter
import json

names_value = config['names']
names = list(names_value.values()) if isinstance(names_value, dict) else names_value
audit = {'classes': names, 'splits': {}}
dataset_root = Path(config['path'])
for split, key in [('train', 'train'), ('val', 'val'), ('test', 'test')]:
    if key not in config:
        continue
    image_dir = dataset_root / config[key]
    label_dir = Path(str(image_dir).replace('/images', '/labels'))
    images = [p for p in image_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
    labels = list(label_dir.rglob('*.txt'))
    counts = Counter()
    invalid = 0
    for label in labels:
        for line in label.read_text(errors='ignore').splitlines():
            fields = line.split()
            if len(fields) != 5:
                invalid += 1
                continue
            counts[int(fields[0])] += 1
    audit['splits'][split] = {
        'images': len(images), 'labels': len(labels), 'invalid_rows': invalid,
        'class_counts': {names[i]: counts[i] for i in range(len(names))}
    }
print(json.dumps(audit, ensure_ascii=False, indent=2))
(OUTPUT_ROOT / 'data_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2))

## Обучение
Fast: YOLO11n/640; Accurate: YOLO11s/768. Аугментации с цветовым сдвигом и mosaic отключены как неестественные для рентгенов. Каждая модель сохраняется в Drive.

In [ ]:
from ultralytics import YOLO
import shutil

def train_profile(name, checkpoint, imgsz, epochs, batch, patience):
    model = YOLO(checkpoint)
    result = model.train(
        data=str(DATA_YAML), epochs=epochs, imgsz=imgsz, batch=batch, patience=patience,
        seed=42, deterministic=True, pretrained=True, device=0, workers=2, amp=True,
        project=str(OUTPUT_ROOT / 'runs'), name=name, exist_ok=True, save_period=5,
        degrees=5.0, translate=0.04, scale=0.10, fliplr=0.5, flipud=0.0,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.12, mosaic=0.0, mixup=0.0, plots=True
    )
    destination = OUTPUT_ROOT / 'models' / f'{name}.pt'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(result.save_dir) / 'weights' / 'best.pt', destination)
    return destination

In [ ]:
FAST_WEIGHTS = train_profile('fast', 'yolo11n.pt', imgsz=640, epochs=35, batch=16, patience=8)
FAST_WEIGHTS

In [ ]:
ACCURATE_WEIGHTS = train_profile('accurate', 'yolo11s.pt', imgsz=768, epochs=50, batch=8, patience=10)
ACCURATE_WEIGHTS

## Финальная оценка на test split и честная таблица метрик

In [ ]:
import pandas as pd

records = []
evaluation_split = 'test' if config.get('test') else 'val'
for name, weights in [('fast', FAST_WEIGHTS), ('accurate', ACCURATE_WEIGHTS)]:
    model = YOLO(str(weights))
    metrics = model.val(
        data=str(DATA_YAML), split=evaluation_split, device=0, plots=True,
        project=str(OUTPUT_ROOT / 'runs' / 'evaluation'), name=name
    )
    records.append({
        'model': name, 'split': evaluation_split,
        'mAP@0.5': float(metrics.box.map50),
        'mAP@0.5:0.95': float(metrics.box.map),
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'inference_ms': float(metrics.speed.get('inference', 0.0)),
        'size_mb': weights.stat().st_size / 1024**2,
        'target_reached': float(metrics.box.map50) >= 0.5
    })
metrics_df = pd.DataFrame(records)
display(metrics_df)
metrics_df.to_csv(OUTPUT_ROOT / 'metrics.csv', index=False)
(OUTPUT_ROOT / 'metrics.json').write_text(metrics_df.to_json(orient='records', force_ascii=False, indent=2))

Готовые файлы находятся в `MyDrive/bone-fracture-detector/`: `models/fast.pt`, `models/accurate.pt`, `metrics.csv`, `metrics.json`, `data_audit.json` и графики в `runs/`. Скопируйте два `.pt` в папку `models/` репозитория, а метрики — в `reports/`.